# 05 — Held-out evaluation and error analysis

Evaluate the preselected detector once on June–August windows. Report event overlap and alert burden only. Because the dataset is unlabeled and reports may be incomplete, alerts outside incidents are not proven false positives and absence of alerts is not a proven false negative. No ROC-AUC, F1 or accuracy is justified.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import RAW, PROCESSED, MODELS, ANALOG, DIGITAL, FAILURES

import joblib
from src.models.predict import load_model,anomaly_scores
from src.evaluation.evaluate import summarize_alerts,event_coverage
test=pd.read_pickle(PROCESSED/'test_features.pkl')
model=load_model(MODELS/'isolation_forest.joblib')
cols=joblib.load(MODELS/'feature_columns.joblib');policy=joblib.load(MODELS/'decision_policy.joblib')
scores=anomaly_scores(model,test[cols]);threshold=policy['threshold']
summary=summarize_alerts(test.timestamp,scores,threshold)
display(pd.DataFrame([summary]));display(event_coverage(test.timestamp,scores,threshold))

## Alert chronology

An alert *during* an incident indicates temporal coincidence, not anticipation. To estimate lead time, independently verified onset and precursor intervals would be required.

In [ ]:
fig,ax=plt.subplots(figsize=(13,4))
ax.plot(test.timestamp,scores,label='Anomaly score',linewidth=.6)
ax.axhline(threshold,color='red',linestyle='--',label='Validation-derived threshold')
for start,end in FAILURES:
    if pd.Timestamp(end)>=test.timestamp.min() and pd.Timestamp(start)<=test.timestamp.max():
        ax.axvspan(pd.Timestamp(start),pd.Timestamp(end),alpha=.2,color='orange')
ax.set(title='Held-out anomaly scores and documented air-leak intervals',xlabel='Time',ylabel='Anomaly score')
ax.legend();plt.tight_layout()

## Error analysis and limitations

Review top-scoring windows and sensor levels below. High alerts outside the four reported intervals may be genuine unreported anomalies, changes in duty cycle, or measurement errors. Feature-level deviation is descriptive rather than causal attribution; SHAP is intentionally omitted to keep the project lean. One compressor and few incidents prevent robust fleet generalization.

In [ ]:
ranked=test.copy();ranked['anomaly_score']=scores
print('Top anomalous windows:')
display(ranked.nlargest(15,'anomaly_score')[['timestamp','anomaly_score']+[c for c in ['TP2_mean','Motor_current_mean','Oil_temperature_mean'] if c in ranked]])
PROCESSED.mkdir(parents=True,exist_ok=True)
pd.DataFrame([summary]).to_csv(PROCESSED/'test_summary.csv',index=False)

## Maintenance implications

Operational usefulness requires an agreed alert budget, technician review, and verified failure-onset records. Next steps: inspect top alerts with domain experts, verify cadence and missing periods, collect labeled maintenance history, and assess event-based detection latency and transfer to other compressors. No deployment-readiness claim is made.